# Validation: GBD 2020 Alabama vs GBD 2023 USA

This notebook compares the old artifact (GBD 2020, Alabama, 1000 draws) with the new artifact (GBD 2023, USA national, 250 draws) and runs the production model to verify correctness.

**Expected differences:**
- Different location (state vs national) -> different absolute rates
- Different GBD round -> updated estimates
- Age bins: 23 -> 25 (added age groups)
- Draw count: 1000 -> 250
- RR format: SBP/LDL-C/FPG switched from log-linear (184 rows) to non-log-linear (200K rows)

**Things that should be similar:**
- Mediation factors (location-independent)
- Exposure distribution weight patterns
- General magnitude of rates and exposures

In [1]:
import pandas as pd
import numpy as np

OLD_PATH = '../src/vivarium_nih_us_cvd/artifacts/alabama.hdf'
NEW_PATH = '../src/vivarium_nih_us_cvd/artifacts/united_states_of_america.hdf'

old_store = pd.HDFStore(OLD_PATH, 'r')
new_store = pd.HDFStore(NEW_PATH, 'r')

print(f'Old artifact keys: {len(old_store.keys())}')
print(f'New artifact keys: {len(new_store.keys())}')
print(f'Keys match: {set(old_store.keys()) == set(new_store.keys())}')

Old artifact keys: 54
New artifact keys: 54
Keys match: True


## 1. Demographic Structure

Compare age bins, population structure, and demographic dimensions.

In [2]:
# Age bins
old_bins = old_store['/population/age_bins']
new_bins = new_store['/population/age_bins']

print(f'Age bins: old={len(old_bins)}, new={len(new_bins)}')
print()

# Find new age groups
old_names = set(old_bins['age_group_name'])
new_names = set(new_bins['age_group_name'])
print('Age groups only in NEW:', new_names - old_names)
print('Age groups only in OLD:', old_names - new_names)
print()

# Population structure (draw_0 values)
old_pop = old_store['/population/structure']
new_pop = new_store['/population/structure']
old_pop.index = old_pop.index.droplevel([c for c in old_pop.index.names if c not in ['age_start', 'age_end', 'sex']])
new_pop.index = new_pop.index.droplevel([c for c in new_pop.index.names if c not in ['age_start', 'age_end', 'sex']])

# Compare common age groups
common = old_pop.index.intersection(new_pop.index)
print(f'Common demographic bins: {len(common)}')
print(f'Old-only demographic bins: {len(old_pop.index.difference(new_pop.index))}')
print(f'New-only demographic bins: {len(new_pop.index.difference(old_pop.index))}')

Age bins: old=23, new=25

Age groups only in NEW: {'12 to 23 months', '1-5 months', '6-11 months', '2 to 4'}
Age groups only in OLD: {'1 to 4', 'Post Neonatal'}

Common demographic bins: 42
Old-only demographic bins: 4
New-only demographic bins: 8


## 2. Risk Exposure Distributions

Compare mean exposure levels for SBP, LDL-C, BMI, and FPG across age and sex. We compare draw_0 values (since draw counts differ).

In [3]:
risks = [
    ('SBP', 'high_systolic_blood_pressure'),
    ('LDL-C', 'high_ldl_cholesterol'),
    ('BMI', 'high_body_mass_index_in_adults'),
    ('FPG', 'high_fasting_plasma_glucose'),
]

for label, risk in risks:
    old_exp = old_store[f'/risk_factor/{risk}/exposure']
    new_exp = new_store[f'/risk_factor/{risk}/exposure']
    
    # Use draw_0 for comparison
    old_vals = old_exp['draw_0'].values
    new_vals = new_exp['draw_0'].values
    
    print(f'{label}:')
    print(f'  Old (Alabama): mean={old_vals.mean():.2f}, '
          f'range=[{old_vals.min():.2f}, {old_vals.max():.2f}], '
          f'rows={len(old_exp)}')
    print(f'  New (USA):     mean={new_vals.mean():.2f}, '
          f'range=[{new_vals.min():.2f}, {new_vals.max():.2f}], '
          f'rows={len(new_exp)}')
    print()

SBP:
  Old (Alabama): mean=84.61, range=[0.00, 146.53], rows=46
  New (USA):     mean=77.50, range=[0.00, 144.83], rows=50

LDL-C:
  Old (Alabama): mean=1.93, range=[0.00, 3.28], rows=46
  New (USA):     mean=1.78, range=[0.00, 3.20], rows=50

BMI:
  Old (Alabama): mean=20.80, range=[0.00, 32.19], rows=46
  New (USA):     mean=18.72, range=[0.00, 30.84], rows=50



FPG:
  Old (Alabama): mean=3.93, range=[0.00, 6.93], rows=46
  New (USA):     mean=3.73, range=[0.00, 6.76], rows=50



In [4]:
# Detailed exposure comparison by age group (ages 25+)
for label, risk in risks:
    old_exp = old_store[f'/risk_factor/{risk}/exposure']
    new_exp = new_store[f'/risk_factor/{risk}/exposure']
    
    # Reset index to get age/sex as columns
    old_df = old_exp.reset_index()
    new_df = new_exp.reset_index()
    
    # Keep only adult age groups (25+)
    old_df = old_df[old_df['age_start'] >= 25]
    new_df = new_df[new_df['age_start'] >= 25]
    
    # Merge on common age/sex bins
    merge_cols = ['age_start', 'age_end', 'sex']
    merged = old_df[merge_cols + ['draw_0']].merge(
        new_df[merge_cols + ['draw_0']],
        on=merge_cols, suffixes=('_old', '_new'),
        how='inner'
    )
    
    merged['pct_diff'] = 100 * (merged['draw_0_new'] - merged['draw_0_old']) / merged['draw_0_old']
    
    print(f'\n{label} — draw_0 comparison (ages 25+):')
    print(f'  Mean pct change: {merged["pct_diff"].mean():.1f}%')
    print(f'  Max abs pct change: {merged["pct_diff"].abs().max():.1f}%')
    
    # Show largest differences
    biggest = merged.nlargest(3, 'pct_diff')[merge_cols + ['draw_0_old', 'draw_0_new', 'pct_diff']]
    smallest = merged.nsmallest(3, 'pct_diff')[merge_cols + ['draw_0_old', 'draw_0_new', 'pct_diff']]
    print('  Largest increases:')
    for _, row in biggest.iterrows():
        pct = row['pct_diff']
        print(f'    age {row["age_start"]:.0f}-{row["age_end"]:.0f} {row["sex"]}: '
              f'{row["draw_0_old"]:.2f} -> {row["draw_0_new"]:.2f} ({pct:+.1f}%)')
    print('  Largest decreases:')
    for _, row in smallest.iterrows():
        pct = row['pct_diff']
        print(f'    age {row["age_start"]:.0f}-{row["age_end"]:.0f} {row["sex"]}: '
              f'{row["draw_0_old"]:.2f} -> {row["draw_0_new"]:.2f} ({pct:+.1f}%)')


SBP — draw_0 comparison (ages 25+):
  Mean pct change: -0.4%
  Max abs pct change: 4.7%
  Largest increases:
    age 25-30 Male: 114.61 -> 118.78 (+3.6%)
    age 80-85 Male: 134.10 -> 138.37 (+3.2%)
    age 60-65 Female: 126.07 -> 129.15 (+2.4%)
  Largest decreases:
    age 35-40 Male: 127.69 -> 121.66 (-4.7%)
    age 40-45 Female: 120.69 -> 116.72 (-3.3%)
    age 45-50 Female: 123.82 -> 119.94 (-3.1%)



LDL-C — draw_0 comparison (ages 25+):
  Mean pct change: 0.5%
  Max abs pct change: 6.6%
  Largest increases:
    age 70-75 Female: 2.99 -> 3.12 (+4.4%)
    age 45-50 Male: 3.01 -> 3.14 (+4.3%)
    age 25-30 Male: 2.85 -> 2.97 (+3.9%)
  Largest decreases:
    age 80-85 Female: 3.28 -> 3.06 (-6.6%)
    age 90-95 Male: 2.78 -> 2.66 (-4.2%)
    age 80-85 Male: 2.77 -> 2.69 (-3.2%)

BMI — draw_0 comparison (ages 25+):
  Mean pct change: -2.1%
  Max abs pct change: 5.0%
  Largest increases:
    age 90-95 Female: 27.13 -> 27.17 (+0.2%)
    age 85-90 Male: 27.49 -> 27.44 (-0.2%)
    age 65-70 Male: 29.96 -> 29.88 (-0.2%)
  Largest decreases:
    age 40-45 Female: 32.19 -> 30.57 (-5.0%)
    age 30-35 Female: 31.64 -> 30.06 (-5.0%)
    age 35-40 Female: 31.89 -> 30.39 (-4.7%)

FPG — draw_0 comparison (ages 25+):
  Mean pct change: 3.5%
  Max abs pct change: 22.4%
  Largest increases:
    age 45-50 Female: 4.86 -> 5.95 (+22.4%)
    age 70-75 Female: 5.46 -> 6.57 (+20.3%)
    age 90-95 Female: 5

## 3. Exposure Distribution Weights

These determine which parametric distribution best fits each risk's exposure in each age/sex bin. They should be structurally similar since distribution shapes are location-independent.

In [5]:
for label, risk in risks:
    old_w = old_store[f'/risk_factor/{risk}/exposure_distribution_weights']
    new_w = new_store[f'/risk_factor/{risk}/exposure_distribution_weights']
    
    print(f'{label}:')
    print(f'  Old shape: {old_w.shape}, New shape: {new_w.shape}')
    
    # Check parameter (distribution type) values
    old_reset = old_w.reset_index()
    new_reset = new_w.reset_index()
    
    if 'parameter' in old_reset.columns:
        old_dists = sorted(old_reset['parameter'].unique())
        print(f'  Old distributions: {old_dists}')
    if 'parameter' in new_reset.columns:
        new_dists = sorted(new_reset['parameter'].unique())
        print(f'  New distributions: {new_dists}')
    print()

SBP:
  Old shape: (598, 1), New shape: (650, 1)
  Old distributions: ['betasr', 'exp', 'gamma', 'glnorm', 'gumbel', 'invgamma', 'invweibull', 'llogis', 'lnorm', 'mgamma', 'mgumbel', 'norm', 'weibull']
  New distributions: ['betasr', 'exp', 'gamma', 'glnorm', 'gumbel', 'invgamma', 'invweibull', 'llogis', 'lnorm', 'mgamma', 'mgumbel', 'norm', 'weibull']

LDL-C:
  Old shape: (598, 1), New shape: (650, 1)
  Old distributions: ['betasr', 'exp', 'gamma', 'glnorm', 'gumbel', 'invgamma', 'invweibull', 'llogis', 'lnorm', 'mgamma', 'mgumbel', 'norm', 'weibull']
  New distributions: ['betasr', 'exp', 'gamma', 'glnorm', 'gumbel', 'invgamma', 'invweibull', 'llogis', 'lnorm', 'mgamma', 'mgumbel', 'norm', 'weibull']



BMI:
  Old shape: (598, 1), New shape: (650, 1)
  Old distributions: ['betasr', 'exp', 'gamma', 'glnorm', 'gumbel', 'invgamma', 'invweibull', 'llogis', 'lnorm', 'mgamma', 'mgumbel', 'norm', 'weibull']
  New distributions: ['betasr', 'exp', 'gamma', 'glnorm', 'gumbel', 'invgamma', 'invweibull', 'llogis', 'lnorm', 'mgamma', 'mgumbel', 'norm', 'weibull']

FPG:
  Old shape: (598, 1), New shape: (650, 1)
  Old distributions: ['betasr', 'exp', 'gamma', 'glnorm', 'gumbel', 'invgamma', 'invweibull', 'llogis', 'lnorm', 'mgamma', 'mgumbel', 'norm', 'weibull']
  New distributions: ['betasr', 'exp', 'gamma', 'glnorm', 'gumbel', 'invgamma', 'invweibull', 'llogis', 'lnorm', 'mgamma', 'mgumbel', 'norm', 'weibull']



## 4. Disease Incidence and Mortality Rates

Compare key cause rates between the two artifacts. Differences here reflect both the location change (Alabama vs USA) and the GBD round update.

In [6]:
rate_keys = [
    ('IS incidence', '/cause/ischemic_stroke/incidence_rate'),
    ('AMI incidence', '/cause/acute_myocardial_infarction/incidence_rate'),
    ('HF-IHD incidence', '/cause/heart_failure_from_ischemic_heart_disease/incidence_rate'),
    ('HF-residual incidence', '/cause/heart_failure_residual/incidence_rate'),
    ('IS CSMR', '/cause/ischemic_stroke/cause_specific_mortality_rate'),
    ('IHD+HF CSMR', '/cause/ischemic_heart_disease_and_heart_failure/cause_specific_mortality_rate'),
    ('All-cause mortality', '/cause/all_causes/cause_specific_mortality_rate'),
]

for label, key in rate_keys:
    old_df = old_store[key].reset_index()
    new_df = new_store[key].reset_index()
    
    # Adults 25+
    old_adult = old_df[old_df['age_start'] >= 25]
    new_adult = new_df[new_df['age_start'] >= 25]
    
    old_mean = old_adult['draw_0'].mean()
    new_mean = new_adult['draw_0'].mean()
    pct = 100 * (new_mean - old_mean) / old_mean if old_mean != 0 else float('inf')
    
    print(f'{label:30s}  old={old_mean:.6f}  new={new_mean:.6f}  change={pct:+.1f}%')

IS incidence                    old=0.004080  new=0.004201  change=+3.0%


AMI incidence                   old=0.010916  new=0.000102  change=-99.1%
HF-IHD incidence                old=0.026973  new=0.023287  change=-13.7%
HF-residual incidence           old=0.021906  new=0.016222  change=-25.9%


IS CSMR                         old=0.003445  new=0.003214  change=-6.7%
IHD+HF CSMR                     old=0.037243  new=0.036493  change=-2.0%


All-cause mortality             old=0.047975  new=0.050217  change=+4.7%


## 5. Relative Risk Comparison

The biggest structural change: SBP, LDL-C, and FPG relative risks switched from log-linear (per-unit RR, ~184 rows) to non-log-linear (exposure-level-specific RRs, ~200K rows). BMI remained log-linear.

We can compare BMI RRs directly and examine the non-log-linear RR structure.

In [7]:
# BMI relative risk — still log-linear in both
old_bmi_rr = old_store['/risk_factor/high_body_mass_index_in_adults/relative_risk'].reset_index()
new_bmi_rr = new_store['/risk_factor/high_body_mass_index_in_adults/relative_risk'].reset_index()

print('BMI Relative Risk')
print(f'  Old shape: {old_bmi_rr.shape}')
print(f'  New shape: {new_bmi_rr.shape}')
print()

# Check affected causes in old vs new
if 'affected_entity' in old_bmi_rr.columns:
    print('Old affected entities:', sorted(old_bmi_rr['affected_entity'].unique()))
if 'affected_entity' in new_bmi_rr.columns:
    print('New affected entities:', sorted(new_bmi_rr['affected_entity'].unique()))
print()

# Compare HF RRs (present in both)
for entity in ['heart_failure_from_ischemic_heart_disease', 'heart_failure_residual']:
    if 'affected_entity' in old_bmi_rr.columns and 'affected_entity' in new_bmi_rr.columns:
        old_hf = old_bmi_rr[old_bmi_rr['affected_entity'] == entity]['draw_0']
        new_hf = new_bmi_rr[new_bmi_rr['affected_entity'] == entity]['draw_0']
        print(f'  BMI -> {entity}:')
        print(f'    Old: mean RR={old_hf.mean():.4f}, range=[{old_hf.min():.4f}, {old_hf.max():.4f}]')
        print(f'    New: mean RR={new_hf.mean():.4f}, range=[{new_hf.min():.4f}, {new_hf.max():.4f}]')

BMI Relative Risk
  Old shape: (276, 1008)
  New shape: (300, 258)

Old affected entities: ['acute_ischemic_stroke', 'acute_myocardial_infarction', 'chronic_ischemic_stroke_to_acute_ischemic_stroke', 'heart_failure_from_ischemic_heart_disease', 'heart_failure_residual', 'post_myocardial_infarction_to_acute_myocardial_infarction']
New affected entities: ['acute_ischemic_stroke', 'acute_myocardial_infarction', 'chronic_ischemic_stroke_to_acute_ischemic_stroke', 'heart_failure_from_ischemic_heart_disease', 'heart_failure_residual', 'post_myocardial_infarction_to_acute_myocardial_infarction']

  BMI -> heart_failure_from_ischemic_heart_disease:
    Old: mean RR=1.0992, range=[1.0000, 1.1342]
    New: mean RR=1.0913, range=[1.0000, 1.1342]
  BMI -> heart_failure_residual:
    Old: mean RR=1.0992, range=[1.0000, 1.1342]
    New: mean RR=1.0913, range=[1.0000, 1.1342]


In [8]:
# Non-log-linear RR structure for SBP (representative)
new_sbp_rr = new_store['/risk_factor/high_systolic_blood_pressure/relative_risk'].reset_index()
old_sbp_rr = old_store['/risk_factor/high_systolic_blood_pressure/relative_risk'].reset_index()

print('SBP Relative Risk')
print(f'  Old: {old_sbp_rr.shape} (log-linear, per-unit RR)')
print(f'  New: {new_sbp_rr.shape} (non-log-linear, exposure-level RRs)')
print()
print('New SBP RR columns:', list(new_sbp_rr.columns[:10]), '...')
print()

if 'exposure' in new_sbp_rr.columns:
    print('Exposure levels in new RR data:')
    print(f'  min={new_sbp_rr["exposure"].min():.1f}, max={new_sbp_rr["exposure"].max():.1f}, '
          f'unique={new_sbp_rr["exposure"].nunique()}')
    print()
    # Show RR at selected exposure levels for one age/sex
    sample = new_sbp_rr[
        (new_sbp_rr['age_start'] == 55) & 
        (new_sbp_rr['sex'] == 'Male')
    ].copy()
    if 'affected_entity' in sample.columns:
        sample = sample[sample['affected_entity'] == 'acute_ischemic_stroke']
    if len(sample) > 0:
        sample = sample.sort_values('exposure')
        print('SBP RR for Male 55-59 on acute IS (draw_0):')
        for _, row in sample.iloc[::max(1, len(sample)//10)].iterrows():
            print(f'  exposure={row["exposure"]:6.1f}  RR={row["draw_0"]:.4f}')

SBP Relative Risk
  Old: (184, 1008) (log-linear, per-unit RR)
  New: (200000, 258) (non-log-linear, exposure-level RRs)

New SBP RR columns: ['sex', 'age_start', 'age_end', 'year_start', 'year_end', 'affected_entity', 'affected_measure', 'parameter', 'draw_0', 'draw_1'] ...



## 6. Joint PAF Comparison

The old artifact stored per-draw PAFs (240 rows x 1000 draws). The new artifact stores precomputed PAFs (192 rows x 8 targets, computed from a dedicated PAF-calculation simulation).

In [9]:
old_paf = old_store['/risk_factor/joint_mediated_risks/population_attributable_fraction'].reset_index()
new_paf = new_store['/risk_factor/joint_mediated_risks/population_attributable_fraction'].reset_index()

print(f'Old PAF: {old_paf.shape}')
print(f'New PAF: {new_paf.shape}')
print()
print('Old PAF columns:', list(old_paf.columns[:8]))
print('New PAF columns:', list(new_paf.columns[:15]))
print()

# New PAF has target columns; show mean PAFs by target
target_cols = [c for c in new_paf.columns if 'incidence_rate' in str(c) or 'transition_rate' in str(c)]
if target_cols:
    print('New PAF mean by target (ages 25+):')
    adult_paf = new_paf[new_paf['age_start'] >= 25]
    for col in sorted(target_cols):
        print(f'  {col:60s}  mean={adult_paf[col].mean():.3f}')
print()

# Old PAF: show mean draw_0 by affected_entity
if 'affected_entity' in old_paf.columns:
    print('Old PAF mean (draw_0) by target (ages 25+):')
    old_adult = old_paf[old_paf['age_start'] >= 25]
    for entity in sorted(old_adult['affected_entity'].unique()):
        subset = old_adult[old_adult['affected_entity'] == entity]
        print(f'  {entity:60s}  mean={subset["draw_0"].mean():.3f}')

Old PAF: (240, 1007)
New PAF: (192, 9)

Old PAF columns: ['sex', 'age_start', 'age_end', 'year_start', 'year_end', 'affected_entity', 'affected_measure', 'draw_0']
New PAF columns: ['index', 'affected_entity', 'affected_measure', 'age_start', 'age_end', 'sex', 'year_start', 'year_end', 'value']


Old PAF mean (draw_0) by target (ages 25+):
  acute_ischemic_stroke                                         mean=0.826
  acute_myocardial_infarction                                   mean=0.873
  chronic_ischemic_stroke_to_acute_ischemic_stroke              mean=0.826
  heart_failure_from_ischemic_heart_disease                     mean=0.360
  heart_failure_residual                                        mean=0.360
  post_myocardial_infarction_to_acute_myocardial_infarction     mean=0.873


## 7. Mediation Factors

Mediation factors should be identical or very close — they represent the proportion of a risk's effect mediated through each pathway and are location-independent.

In [10]:
old_mf = old_store['/risk/cause/mediation_factors'].reset_index()
new_mf = new_store['/risk/cause/mediation_factors'].reset_index()

print(f'Old mediation factors: {old_mf.shape}')
print(f'New mediation factors: {new_mf.shape}')
print()

# Compare values
merge_cols = [c for c in old_mf.columns if c != 'value' and c in new_mf.columns]
comparison = old_mf.merge(new_mf, on=merge_cols, suffixes=('_old', '_new'))

if 'value_old' in comparison.columns and 'value_new' in comparison.columns:
    comparison['diff'] = comparison['value_new'] - comparison['value_old']
    print(f'Max absolute difference: {comparison["diff"].abs().max():.6f}')
    print(f'All identical: {(comparison["diff"].abs() < 1e-10).all()}')
    print()
    if comparison['diff'].abs().max() > 1e-10:
        print('Differences:')
        print(comparison[comparison['diff'].abs() > 1e-10].to_string())
    else:
        print('Mediation factors match exactly (location-independent, as expected).')
else:
    print('Columns:', list(comparison.columns))

Old mediation factors: (16, 1004)
New mediation factors: (16, 1004)



Columns: ['risk_name', 'mediator_name', 'affected_entity', 'draw_0', 'draw_1', 'draw_2', 'draw_3', 'draw_4', 'draw_5', 'draw_6', 'draw_7', 'draw_8', 'draw_9', 'draw_10', 'draw_11', 'draw_12', 'draw_13', 'draw_14', 'draw_15', 'draw_16', 'draw_17', 'draw_18', 'draw_19', 'draw_20', 'draw_21', 'draw_22', 'draw_23', 'draw_24', 'draw_25', 'draw_26', 'draw_27', 'draw_28', 'draw_29', 'draw_30', 'draw_31', 'draw_32', 'draw_33', 'draw_34', 'draw_35', 'draw_36', 'draw_37', 'draw_38', 'draw_39', 'draw_40', 'draw_41', 'draw_42', 'draw_43', 'draw_44', 'draw_45', 'draw_46', 'draw_47', 'draw_48', 'draw_49', 'draw_50', 'draw_51', 'draw_52', 'draw_53', 'draw_54', 'draw_55', 'draw_56', 'draw_57', 'draw_58', 'draw_59', 'draw_60', 'draw_61', 'draw_62', 'draw_63', 'draw_64', 'draw_65', 'draw_66', 'draw_67', 'draw_68', 'draw_69', 'draw_70', 'draw_71', 'draw_72', 'draw_73', 'draw_74', 'draw_75', 'draw_76', 'draw_77', 'draw_78', 'draw_79', 'draw_80', 'draw_81', 'draw_82', 'draw_83', 'draw_84', 'draw_85', 'draw

## 8. Medication and Treatment Data

Compare medication adherence exposure and LDL-C medication effect.

In [11]:
# Medication adherence
for med in ['sbp_medication_adherence', 'ldlc_medication_adherence']:
    old_df = old_store[f'/risk_factor/{med}/exposure'].reset_index()
    new_df = new_store[f'/risk_factor/{med}/exposure'].reset_index()
    print(f'{med}:')
    print(f'  Old: {old_df.shape}')
    print(f'  New: {new_df.shape}')
    
    # Show adherence levels
    if 'parameter' in old_df.columns:
        for param in sorted(old_df['parameter'].unique()):
            old_subset = old_df[old_df['parameter'] == param]
            new_subset = new_df[new_df['parameter'] == param] if param in new_df['parameter'].values else pd.DataFrame()
            old_val = old_subset['draw_0'].mean()
            new_val = new_subset['draw_0'].mean() if len(new_subset) > 0 else float('nan')
            print(f'    {param}: old={old_val:.4f}, new={new_val:.4f}')
    print()

# LDL-C medication effect
old_me = old_store['/risk_factor/high_ldl_cholesterol/medication_effect'].reset_index()
new_me = new_store['/risk_factor/high_ldl_cholesterol/medication_effect'].reset_index()
print('LDL-C medication effect:')
print(f'  Old: {old_me.shape}')
print(f'  New: {new_me.shape}')
print()
print('Old columns:', list(old_me.columns[:8]))
print('New columns:', list(new_me.columns[:8]))

sbp_medication_adherence:
  Old: (138, 1007)
  New: (150, 257)
    cat1: old=0.1600, new=0.1600
    cat2: old=0.1008, new=0.1008
    cat3: old=0.7392, new=0.7392

ldlc_medication_adherence:
  Old: (138, 1007)
  New: (150, 257)
    cat1: old=0.2500, new=0.2500
    cat2: old=0.0975, new=0.0975
    cat3: old=0.6525, new=0.6525

LDL-C medication effect:
  Old: (5, 1001)
  New: (50, 255)

Old columns: ['ldlc_medication', 'draw_0', 'draw_1', 'draw_2', 'draw_3', 'draw_4', 'draw_5', 'draw_6']
New columns: ['sex', 'age_start', 'age_end', 'year_start', 'year_end', 'draw_0', 'draw_1', 'draw_2']


## 9. Production Model Run

Run the production model specification for 1 year (13 x 28-day steps) and inspect results.

In [12]:
old_store.close()
new_store.close()

from vivarium import InteractiveContext

yaml_path = '../src/vivarium_nih_us_cvd/model_specifications/nih_us_cvd.yaml'
sim = InteractiveContext(yaml_path, setup=False)
sim.configuration.update({'population': {'population_size': 2_000}})
sim.setup()
print('Setup complete.')

Setup complete.


In [13]:
# Run 13 steps (~1 year)
for i in range(13):
    sim.step()
print(f'Simulated 13 steps (364 days).')

Simulated 13 steps (364 days).


In [14]:
pop = sim.get_population([
    'is_alive', 'age', 'sex',
    'ischemic_stroke',
    'ischemic_heart_disease_and_heart_failure',
])

print(f'Population: {len(pop)} simulants')
print(f'Alive: {pop["is_alive"].sum()} ({100*pop["is_alive"].mean():.1f}%)')
print(f'Deaths: {(~pop["is_alive"]).sum()}')
print()

for col in ['ischemic_stroke', 'ischemic_heart_disease_and_heart_failure']:
    print(f'{col}:')
    counts = pop[col].value_counts()
    for state, count in counts.items():
        print(f'  {state}: {count} ({100*count/len(pop):.1f}%)')
    print()

Population: 2000 simulants
Alive: 1983 (99.2%)
Deaths: 17

ischemic_stroke:
  susceptible_to_ischemic_stroke: 1954 (97.7%)
  chronic_ischemic_stroke: 46 (2.3%)

ischemic_heart_disease_and_heart_failure:
  susceptible_to_ischemic_heart_disease_and_heart_failure: 1957 (97.8%)
  heart_failure_residual: 21 (1.1%)
  heart_failure_from_ischemic_heart_disease: 19 (0.9%)
  post_myocardial_infarction: 3 (0.1%)



In [15]:
risk_cols = [
    'high_systolic_blood_pressure.exposure',
    'high_ldl_cholesterol.exposure',
    'high_body_mass_index_in_adults.exposure',
    'high_fasting_plasma_glucose.exposure',
]

risk_pop = sim.get_population(risk_cols + ['age', 'sex', 'is_alive'])
alive = risk_pop[risk_pop['is_alive']]

print('Risk exposure summary (alive simulants):')
print('=' * 70)
for col in risk_cols:
    vals = alive[col]
    short = col.replace('.exposure', '')
    print(f'{short:45s}  mean={vals.mean():.2f}  std={vals.std():.2f}  '
          f'[{vals.min():.1f}, {vals.max():.1f}]')

Risk exposure summary (alive simulants):
high_systolic_blood_pressure                   mean=105.17  std=36.51  [47.4, 196.7]
high_ldl_cholesterol                           mean=2.16  std=1.51  [0.0, 6.9]
high_body_mass_index_in_adults                 mean=24.84  std=11.33  [5.0, 55.0]
high_fasting_plasma_glucose                    mean=4.54  std=2.52  [1.0, 12.9]


In [16]:
tx_cols = [
    'sbp_medication', 'ldlc_medication',
    'sbp_medication_adherence', 'ldlc_medication_adherence',
]

tx_pop = sim.get_population(tx_cols + ['is_alive'])
alive_tx = tx_pop[tx_pop['is_alive']]

print('Treatment status (alive simulants):')
print('=' * 70)
for col in ['sbp_medication', 'ldlc_medication']:
    print(f'\n{col}:')
    counts = alive_tx[col].value_counts()
    for val, count in counts.items():
        print(f'  {val}: {count} ({100*count/len(alive_tx):.1f}%)')
print()
for col in ['sbp_medication_adherence', 'ldlc_medication_adherence']:
    print(f'{col}:')
    counts = alive_tx[col].value_counts()
    for val, count in counts.items():
        print(f'  {val}: {count} ({100*count/len(alive_tx):.1f}%)')
    print()

Treatment status (alive simulants):

sbp_medication:
  no_treatment: 1478 (74.5%)
  one_drug_half_dose_efficacy: 331 (16.7%)
  two_drug_half_dose_efficacy: 130 (6.6%)
  one_drug_std_dose_efficacy: 30 (1.5%)
  two_drug_std_dose_efficacy: 11 (0.6%)
  three_drug_half_dose_efficacy: 2 (0.1%)
  three_drug_std_dose_efficacy: 1 (0.1%)

ldlc_medication:
  no_treatment: 1674 (84.4%)
  medium_intensity: 154 (7.8%)
  low_intensity: 102 (5.1%)
  high_intensity: 50 (2.5%)
  low_med_with_eze: 3 (0.2%)

sbp_medication_adherence:
  cat3: 1440 (72.6%)
  cat1: 338 (17.0%)
  cat2: 205 (10.3%)

ldlc_medication_adherence:
  cat3: 1332 (67.2%)
  cat1: 456 (23.0%)
  cat2: 195 (9.8%)



In [17]:
results = sim.get_results()
print(f'Total result categories: {len(results)}')
print()

# Key metrics
for key in ['deaths', 'ylls', 'ylds']:
    if key in results:
        df = results[key]
        total = df['value'].sum()
        print(f'{key}: {total:.1f}')

print()

# Disease transitions
for key in sorted(results.keys()):
    if 'transition_count' in key:
        df = results[key]
        total = df['value'].sum()
        print(f'{key}: {total:.0f} transitions')

print()

# Healthcare visits
for key in sorted(results.keys()):
    if 'healthcare_visits' in key:
        df = results[key]
        total = df['value'].sum()
        print(f'{key}: {total:.0f}')

Total result categories: 39

deaths: 17.0
ylls: 401.6
ylds: 24.5

transition_count_ischemic_heart_disease_and_heart_failure: 16 transitions
transition_count_ischemic_stroke: 6 transitions

healthcare_visits_background: 6697
healthcare_visits_emergency: 3
healthcare_visits_missed: 99
healthcare_visits_none: 18109
healthcare_visits_scheduled: 975


## Summary

### Expected differences (GBD round + location change)
- Absolute rate levels differ (Alabama vs USA national)
- Age bins expanded (23 -> 25 groups)
- SBP/LDL-C/FPG RRs switched to non-log-linear format (200K rows)
- Joint PAF format changed (per-draw -> precomputed)

### Should be preserved
- Mediation factors (location-independent)
- Exposure distribution weight structure (13 distributions per bin)
- General simulation behavior: disease transitions, treatment uptake, mortality

### Known limitations
- BMI -> IS/MI relative risks are stubs (RR=1.0)
- PAFs will need recomputation after real BMI data is added